# EO4040 — Tarea 1: Calidad y limpieza de datos
## Dataset: ENIGH 2022 — concentradohogar

**Valor total:** 80 pts automáticos + 20 pts presentación en clase

---

### Instrucciones generales

1. Descarga el archivo `concentradohogar.csv` desde: https://www.inegi.org.mx/programas/enigh/nc/2022/
2. Coloca el archivo en la **misma carpeta que este notebook**
3. Corre todas las celdas **en orden, de arriba a abajo**, sin saltar ninguna
4. **Los nombres de las variables importan.** Cada pregunta indica el nombre exacto que debe tener tu variable — si la llamas diferente, el calificador automático no la encontrará y la pregunta quedará en cero
5. El dataset original (`df`) **nunca debe modificarse** — toda limpieza se hace sobre copias
6. Al terminar, exporta tu entrega con la celda al final del notebook

---

In [ ]:
# Celda de configuración — no modificar
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('✅ Librerías cargadas correctamente')

---
## Parte 1 — Cargar el dataset
**10 puntos**

### Pregunta 1 — Cargar el archivo (10 pts)

Carga el archivo `concentradohogar.csv` en un DataFrame.

**Requisitos:**
- Usa `encoding='latin-1'` para evitar errores con caracteres especiales del español
- El DataFrame debe llamarse exactamente **`df`**

El calificador verifica que:
- `df` exista
- Tenga al menos 70,000 filas (la ENIGH 2022 tiene ~89,000 hogares)
- Tenga al menos 100 columnas
- Contenga la columna `folioviv` (identificador del hogar)

In [ ]:
df = pd.read_csv('concentradohogar.csv', encoding='latin-1')  # SOLUTION
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')

In [ ]:
def test_q1_existe(env):
    """La variable df debe existir"""
    assert 'df' in env, "La variable 'df' no existe. ¿Ejecutaste la celda de carga?"
test_q1_existe(globals())

In [ ]:
def test_q1_filas(df):
    """El dataset debe tener al menos 70,000 filas"""
    assert len(df) >= 70_000, (
        f"Se esperaban ≥70,000 filas, se encontraron {len(df)}. "
        "¿Cargaste el archivo correcto?"
    )
test_q1_filas(df)

In [ ]:
def test_q1_columnas(df):
    """El dataset debe tener al menos 100 columnas"""
    assert df.shape[1] >= 100, (
        f"Se esperaban ≥100 columnas, se encontraron {df.shape[1]}"
    )
test_q1_columnas(df)

In [ ]:
def test_q1_folioviv(df):
    """La columna 'folioviv' debe existir"""
    assert 'folioviv' in df.columns, (
        "La columna 'folioviv' no existe. ¿Usaste el archivo correcto?"
    )
test_q1_folioviv(df)

---
## Parte 2 — Diagnóstico de calidad
**20 puntos**

### Pregunta 2 — Tabla de calidad (10 pts)

Construye un resumen de calidad del dataset.

**La variable debe llamarse exactamente `tabla_calidad`** y ser un DataFrame con estas 4 columnas en este orden:

| Columna | Contenido |
|---|---|
| `variable` | Nombre de cada columna del dataset |
| `tipo_dato` | Tipo de dato en Python — usa `str(df[col].dtype)` |
| `pct_faltantes` | Porcentaje de valores faltantes (0.0 a 100.0), redondeado a 2 decimales |
| `n_unicos` | Número de valores únicos sin contar NaN — usa `.nunique()` |

Debe tener una fila por cada columna del dataset original (todas, no solo algunas).

In [ ]:
# BEGIN SOLUTION
tabla_calidad = pd.DataFrame({
    'variable':      df.columns.tolist(),
    'tipo_dato':     [str(df[c].dtype) for c in df.columns],
    'pct_faltantes': [(df[c].isnull().sum() / len(df) * 100).round(2) for c in df.columns],
    'n_unicos':      [df[c].nunique() for c in df.columns],
})
# END SOLUTION
tabla_calidad.head(10)

In [ ]:
def test_q2_existe(env):
    """La variable tabla_calidad debe existir"""
    assert 'tabla_calidad' in env, "La variable 'tabla_calidad' no existe"
test_q2_existe(globals())

In [ ]:
def test_q2_columnas_requeridas(tabla_calidad):
    """tabla_calidad debe tener las 4 columnas requeridas"""
    requeridas = {'variable', 'tipo_dato', 'pct_faltantes', 'n_unicos'}
    faltantes = requeridas - set(tabla_calidad.columns)
    assert not faltantes, f"Faltan estas columnas en tabla_calidad: {faltantes}"
test_q2_columnas_requeridas(tabla_calidad)

In [ ]:
def test_q2_todas_las_variables(df, tabla_calidad):
    """tabla_calidad debe tener una fila por cada columna del dataset"""
    assert len(tabla_calidad) == len(df.columns), (
        f"Se esperaban {len(df.columns)} filas (una por columna del dataset), "
        f"se encontraron {len(tabla_calidad)}"
    )
test_q2_todas_las_variables(df, tabla_calidad)

In [ ]:
def test_q2_rango_pct(tabla_calidad):
    """Los porcentajes de faltantes deben estar entre 0 y 100"""
    assert tabla_calidad['pct_faltantes'].between(0, 100).all(), (
        "La columna pct_faltantes tiene valores fuera del rango [0, 100]. "
        "¿Multiplicaste por 100?"
    )
test_q2_rango_pct(tabla_calidad)

### Pregunta 3 — Identificar problemas (10 pts)

Con base en tu `tabla_calidad`, crea las siguientes tres variables con **exactamente estos nombres**:

| Variable | Tipo | Contenido |
|---|---|---|
| `vars_con_faltantes` | `list` | Nombres de columnas con más de 0% de faltantes |
| `n_duplicados` | `int` | Número de filas con `folioviv` duplicado |
| `col_mas_faltantes` | `str` | Nombre de la columna con el mayor porcentaje de faltantes |

In [ ]:
# BEGIN SOLUTION
vars_con_faltantes = tabla_calidad[tabla_calidad['pct_faltantes'] > 0]['variable'].tolist()
n_duplicados       = df.duplicated(subset='folioviv').sum()
col_mas_faltantes  = tabla_calidad.loc[tabla_calidad['pct_faltantes'].idxmax(), 'variable']
# END SOLUTION
print(f'Variables con faltantes: {len(vars_con_faltantes)}')
print(f'Duplicados en folioviv:  {n_duplicados}')
print(f'Columna con más faltantes: {col_mas_faltantes}')

In [ ]:
def test_q3_tipo_lista(env):
    """vars_con_faltantes debe ser una lista"""
    assert isinstance(env.get('vars_con_faltantes'), list), (
        "'vars_con_faltantes' debe ser de tipo list. "
        "Si usaste .tolist() al final deberías estar bien."
    )
test_q3_tipo_lista(globals())

In [ ]:
def test_q3_vars_correctas(df, vars_con_faltantes):
    """vars_con_faltantes debe contener exactamente las columnas con NaN"""
    esperadas = set(c for c in df.columns if df[c].isnull().sum() > 0)
    obtenidas = set(vars_con_faltantes)
    assert obtenidas == esperadas, (
        f"Diferencia entre lo esperado y lo obtenido: {obtenidas.symmetric_difference(esperadas)}"
    )
test_q3_vars_correctas(df, vars_con_faltantes)

In [ ]:
def test_q3_duplicados(df, n_duplicados):
    """n_duplicados debe ser el conteo correcto"""
    esperado = int(df.duplicated(subset='folioviv').sum())
    assert int(n_duplicados) == esperado, (
        f"Se esperaban {esperado} duplicados, se obtuvo {n_duplicados}"
    )
test_q3_duplicados(df, n_duplicados)

In [ ]:
def test_q3_col_mas_faltantes(df, col_mas_faltantes):
    """col_mas_faltantes debe ser la columna con mayor % de NaN"""
    esperada = df.isnull().mean().idxmax()
    assert col_mas_faltantes == esperada, (
        f"Se esperaba '{esperada}', se obtuvo '{col_mas_faltantes}'"
    )
test_q3_col_mas_faltantes(df, col_mas_faltantes)

---
## Parte 3 — Limpieza
**30 puntos**

### Pregunta 4 — Crear copia y eliminar duplicados (10 pts)

**La variable debe llamarse exactamente `df_clean`.**

1. Crea una copia del dataset original con `.copy()` y guárdala en `df_clean`
2. Sobre `df_clean`, elimina los registros con `folioviv` duplicado conservando la primera aparición

El dataset original `df` **no debe modificarse** — el calificador lo verifica.

> **¿Por qué eliminar y no conservar ambos?** En la ENIGH, `folioviv` identifica de forma única a cada hogar. Un duplicado es un error de captura o de procesamiento, no un hogar real distinto.

In [ ]:
# BEGIN SOLUTION
df_clean = df.copy()
df_clean = df_clean.drop_duplicates(subset='folioviv', keep='first')
# END SOLUTION
print(f'Filas originales:                  {len(df):,}')
print(f'Filas después de quitar duplicados: {len(df_clean):,}')

In [ ]:
def test_q4_raw_intacto(df, df_clean):
    """El dataset original df no debe haberse modificado"""
    assert id(df) != id(df_clean), (
        "df_clean parece ser el mismo objeto que df — usa df.copy()"
    )
    assert len(df) >= 70_000, "El dataset original df fue modificado"
test_q4_raw_intacto(df, df_clean)

In [ ]:
def test_q4_sin_duplicados(df_clean):
    """df_clean no debe tener folioviv duplicados"""
    n_dup = df_clean.duplicated(subset='folioviv').sum()
    assert n_dup == 0, f"Aún quedan {n_dup} duplicados en folioviv"
test_q4_sin_duplicados(df_clean)

In [ ]:
def test_q4_filas_reducidas(df, df_clean):
    """df_clean debe tener igual o menos filas que df"""
    assert len(df_clean) <= len(df), (
        f"df_clean tiene más filas ({len(df_clean)}) que df ({len(df)})"
    )
test_q4_filas_reducidas(df, df_clean)

### Pregunta 5 — Imputar ingreso corriente (10 pts)

La variable `ingcor` es el ingreso corriente trimestral del hogar. Tiene valores faltantes.

Sobre `df_clean`, realiza los siguientes dos pasos **en este orden**:

1. Crea la columna **`ingcor_faltante`**: vale `1` si `ingcor` era NaN antes de imputar, `0` si no
2. Imputa los NaN de **`ingcor`** con la mediana de la columna

> **¿Por qué la mediana y no la media?** La distribución del ingreso en México es muy asimétrica — hay hogares con ingresos extremadamente altos que jalan la media hacia arriba. La mediana es más representativa del hogar típico y no se ve afectada por esos valores extremos.

In [ ]:
# BEGIN SOLUTION
df_clean['ingcor_faltante'] = df_clean['ingcor'].isnull().astype(int)
mediana_ingcor = df_clean['ingcor'].median()
df_clean['ingcor'] = df_clean['ingcor'].fillna(mediana_ingcor)
# END SOLUTION
print(f'Valores faltantes en ingcor después de imputar: {df_clean["ingcor"].isnull().sum()}')
print(f'Hogares con ingreso imputado (ingcor_faltante=1): {df_clean["ingcor_faltante"].sum()}')

In [ ]:
def test_q5_sin_nan(df_clean):
    """ingcor no debe tener NaN después de la imputación"""
    n = df_clean['ingcor'].isnull().sum()
    assert n == 0, f"Quedan {n} NaN en ingcor — ¿aplicaste fillna()?"
test_q5_sin_nan(df_clean)

In [ ]:
def test_q5_indicador_existe(df_clean):
    """La columna ingcor_faltante debe existir en df_clean"""
    assert 'ingcor_faltante' in df_clean.columns, (
        "La columna 'ingcor_faltante' no existe. "
        "Recuerda crearla ANTES de imputar."
    )
test_q5_indicador_existe(df_clean)

In [ ]:
def test_q5_indicador_binario(df_clean):
    """ingcor_faltante debe contener solo 0 y 1"""
    valores = set(df_clean['ingcor_faltante'].unique())
    assert valores.issubset({0, 1}), (
        f"ingcor_faltante contiene valores distintos de 0 y 1: {valores}"
    )
test_q5_indicador_binario(df_clean)

In [ ]:
def test_q5_conteo_consistente(df, df_clean):
    """El número de 1s en ingcor_faltante debe coincidir con los NaN originales en df"""
    n_nan_original = int(df['ingcor'].isnull().sum())
    n_imputados    = int(df_clean['ingcor_faltante'].sum())
    assert n_imputados == n_nan_original, (
        f"El dataset original tenía {n_nan_original} NaN en ingcor, "
        f"pero ingcor_faltante tiene {n_imputados} unos. "
        "¿Creaste el indicador antes de imputar?"
    )
test_q5_conteo_consistente(df, df_clean)

### Pregunta 6 — Corregir el tipo de `est_socio` (10 pts)

La variable `est_socio` indica el estrato socioeconómico del hogar:
- 1 = Bajo
- 2 = Medio bajo
- 3 = Medio alto
- 4 = Alto

Tiene un orden natural — es una variable **ordinal**, no solo nominal. Pandas no lo sabe a menos que se lo indiquemos.

Convierte `df_clean['est_socio']` a tipo categórico ordenado usando:
```python
pd.CategoricalDtype(categories=[1, 2, 3, 4], ordered=True)
```

> **¿Por qué importa?** Con `ordered=True`, Pandas puede hacer comparaciones como `est_socio > 2` y los modelos de ML pueden usar correctamente el orden de los niveles.

In [ ]:
# BEGIN SOLUTION
cat_type = pd.CategoricalDtype(categories=[1, 2, 3, 4], ordered=True)
df_clean['est_socio'] = df_clean['est_socio'].astype(cat_type)
# END SOLUTION
print(f'Tipo de est_socio: {df_clean["est_socio"].dtype}')
print(f'¿Ordenada?: {df_clean["est_socio"].cat.ordered}')
print(df_clean['est_socio'].value_counts().sort_index())

In [ ]:
def test_q6_es_category(df_clean):
    """est_socio debe ser de tipo category"""
    assert str(df_clean['est_socio'].dtype) == 'category', (
        f"est_socio es de tipo '{df_clean['est_socio'].dtype}', se esperaba 'category'"
    )
test_q6_es_category(df_clean)

In [ ]:
def test_q6_es_ordenada(df_clean):
    """est_socio debe ser una categoría ordenada"""
    assert df_clean['est_socio'].cat.ordered, (
        "est_socio es categórica pero NO ordenada. "
        "Usa ordered=True en CategoricalDtype."
    )
test_q6_es_ordenada(df_clean)

In [ ]:
def test_q6_categorias_correctas(df_clean):
    """Las categorías deben ser [1, 2, 3, 4] en ese orden"""
    cats = list(df_clean['est_socio'].cat.categories)
    assert cats == [1, 2, 3, 4], (
        f"Se esperaban categorías [1, 2, 3, 4], se encontraron {cats}"
    )
test_q6_categorias_correctas(df_clean)

---
## Parte 4 — Verificación
**20 puntos**

### Pregunta 7 — Reporte comparativo (10 pts)

Crea un DataFrame llamado **`reporte`** que compare el dataset original con el limpio.

Debe tener exactamente estas 3 columnas y 2 filas:

| `dataset` | `n_filas` | `total_nan` |
|---|---|---|
| `'original'` | número de filas de `df` | suma de todos los NaN en `df` |
| `'limpio'` | número de filas de `df_clean` | suma de todos los NaN en `df_clean` |

Para calcular el total de NaN en un DataFrame: `df.isnull().sum().sum()`

In [ ]:
# BEGIN SOLUTION
reporte = pd.DataFrame({
    'dataset':   ['original', 'limpio'],
    'n_filas':   [len(df), len(df_clean)],
    'total_nan': [df.isnull().sum().sum(), df_clean.isnull().sum().sum()],
})
# END SOLUTION
reporte

In [ ]:
def test_q7_estructura(env):
    """reporte debe tener las 3 columnas y 2 filas requeridas"""
    r = env.get('reporte')
    assert r is not None, "La variable 'reporte' no existe"
    assert set(r.columns) == {'dataset', 'n_filas', 'total_nan'}, (
        f"Columnas incorrectas: {list(r.columns)}"
    )
    assert len(r) == 2, f"reporte debe tener 2 filas, tiene {len(r)}"
test_q7_estructura(globals())

In [ ]:
def test_q7_valores(df, df_clean, reporte):
    """Los valores de n_filas y total_nan deben ser correctos"""
    r = reporte.set_index('dataset')
    assert int(r.loc['original', 'n_filas']) == len(df), (
        f"n_filas de 'original' incorrecto: se obtuvo {r.loc['original','n_filas']}, se esperaba {len(df)}"
    )
    assert int(r.loc['limpio', 'n_filas']) == len(df_clean), (
        f"n_filas de 'limpio' incorrecto: se obtuvo {r.loc['limpio','n_filas']}, se esperaba {len(df_clean)}"
    )
    assert int(r.loc['original', 'total_nan']) == int(df.isnull().sum().sum()), (
        "total_nan de 'original' es incorrecto"
    )
test_q7_valores(df, df_clean, reporte)

In [ ]:
def test_q7_mejoro(reporte):
    """El dataset limpio debe tener igual o menos NaN que el original"""
    r = reporte.set_index('dataset')
    assert r.loc['limpio', 'total_nan'] <= r.loc['original', 'total_nan'], (
        "El dataset 'limpio' tiene MÁS NaN que el original — revisa tu limpieza"
    )
test_q7_mejoro(reporte)

### Pregunta 8 — Guardar el dataset limpio (10 pts)

1. Guarda `df_clean` en un archivo CSV llamado **`enigh2022_limpio.csv`** sin el índice de Pandas
2. Vuelve a cargarlo en una variable llamada **`df_verificacion`**
3. El calificador verifica que el archivo existe y que `df_verificacion` tiene el mismo número de filas que `df_clean`

In [ ]:
# BEGIN SOLUTION
df_clean.to_csv('enigh2022_limpio.csv', index=False)
df_verificacion = pd.read_csv('enigh2022_limpio.csv')
# END SOLUTION
print(f'Archivo guardado: {df_verificacion.shape[0]:,} filas × {df_verificacion.shape[1]} columnas')

In [ ]:
def test_q8_archivo_existe(env):
    """El archivo enigh2022_limpio.csv debe existir y df_verificacion estar cargado"""
    import os
    assert os.path.exists('enigh2022_limpio.csv'), (
        "El archivo 'enigh2022_limpio.csv' no existe. ¿Ejecutaste to_csv()?"
    )
    assert 'df_verificacion' in env, "La variable 'df_verificacion' no existe"
test_q8_archivo_existe(globals())

In [ ]:
def test_q8_dimensiones(df_clean, df_verificacion):
    """df_verificacion debe tener las mismas filas que df_clean"""
    assert df_verificacion.shape[0] == df_clean.shape[0], (
        f"df_clean tiene {df_clean.shape[0]} filas pero "
        f"df_verificacion tiene {df_verificacion.shape[0]}"
    )
test_q8_dimensiones(df_clean, df_verificacion)

---
## Presentación en clase (20 pts)

Esta parte no se califica aquí — la presentarás en clase en **5 minutos**.

Prepárate para responder cualquiera de estas preguntas sin ver el notebook:

1. ¿Cuántas variables tenían faltantes? ¿Cuál tenía más? ¿Por qué crees que falta ese dato?
2. ¿Por qué usaste la mediana para imputar `ingcor` y no la media?
3. Si los hogares con ingreso muy alto o muy bajo son los que más faltan en `ingcor` (lo que se llama MNAR), ¿qué sesgo introduce tu imputación con mediana?
4. ¿Cómo afectaría ese sesgo a un análisis de desigualdad de ingresos en México?

---
## Exportar tu entrega

Corre la siguiente celda para generar el archivo `.zip` que subirás a Canvas.